# 🏗️ Hunyuan3D-2.1 Architectural Shape & CAD/Fabrication Pipeline (Kaggle)
단 한 장의 2D 건축 컨셉 디자인 스케치(또는 4/6방향 턴어라운드 시트)로부터 디지털 제작(3D 프린팅, CNC 밀링, 와플 그리드 레이저 커팅) 및 CAD 설계가 즉시 가능한 정밀 정규화 기하학 데이터(STL, PLY, quad-friendly OBJ, DXF/SVG 단면 슬라이스)를 복원하는 전용 경량형 초고속 파이프라인입니다.

## 📐 건축/제작 특화 특징 (Traditional vs. Specialized CAD)
1. **경량성 및 속도 극대화**: 무겁고 오래 걸리는 텍스처(디퓨전 Paint) 단계를 완전히 생략하여 **VRAM을 10GB 미만으로 절약**하고 복원 시간을 단 **20초** 내외로 단축합니다.
2. **지면 고정 장치 (Ground Locking)**: 3D 복원물의 둥근 바닥 부분을 지정된 비율(기본 5%)만큼 깔끔하게 수평으로 평평하게 잘라내고, 바닥면을 수치 기준 원점($Z=0$)으로 완벽히 정렬하여 지면에 밀착시킵니다.
3. **정밀 노이즈 스무딩 (Laplacian & Taubin)**: 신경망 메쉬의 조밀한 복셀/마칭 큐브 노이즈를 스무딩 필터로 해결하여 정밀 부재 가공이 가능한 미려한 곡선 표면을 추출합니다. Taubin 알고리즘을 사용하면 메쉬 부피(Volume)의 수축 없이 고주파 노이즈만 제거할 수 있습니다.
4. **수평 등고선 단면 슬라이싱 (DXF/SVG Contours)**: 사용자가 지정한 물리 간격(예: 0.2m)마다 가로 단면을 슬라이싱하여 모든 레이어가 결합된 3D DXF 파일과 각 레이저 커팅용 2D SVG 드로잉들을 자동 추출합니다.

## 🚀 실행 전 Kaggle 세션 설정 필수 요건
1. **GPU 활성화**: 우측 상단 **Session options** → **Accelerator** → **GPU T4 x2** 또는 **GPU T4 x1** 선택 (**필수**)
2. **인터넷 연결**: 우측 상단 **Session options** → **Internet** → **On** 설정 (**외부 GitHub 및 모델 다운로드를 위해 필수**)
3. **스케치 이미지 업로드**: 우측 상단 **Add Data** 버튼을 통해 본인의 2D 빌딩 컨셉 이미지 또는 캐릭터/건물 시트를 업로드하십시오. (또는 Kaggle 기본 아무 데이터셋 추가 가능)
   - 추가한 이미지/데이터셋은 `/kaggle/input/<dataset-slug>/` 경로에 배치됩니다.

In [ ]:
# 1. GPU 및 환경 진단
!nvidia-smi

In [ ]:
# 2. 커스텀 리포지토리 설정
# 본인의 원격 GitHub 레포지토리 주소로 변경하여 최신 코드를 연동할 수 있습니다.
GITHUB_REPO_URL = "https://github.com/jskim062/image_to_3d.git"
print(f"대상 GitHub Repository: {GITHUB_REPO_URL}")

In [ ]:
# 3. Tencent Hunyuan3D-2.1 공식 리포지토리 및 건축 파이프라인 의존성 구축
import os, glob, sys, shutil
import torch

REPO_DIR = '/kaggle/working/Hunyuan3D-2.1'
CUSTOM_DIR = '/kaggle/working/image_to_3d'

# A. Tencent Hunyuan3D-2.1 클론
if not os.path.exists(REPO_DIR):
    print('[*] Tencent Hunyuan3D-2.1 공식 저장소 클론 중...')
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git {REPO_DIR}
else:
    print('[*] Tencent Hunyuan3D-2.1 공식 저장소 이미 존재')

# B. 본인 GitHub Custom Repository 최신 코드 다운로드 및 병합
if os.path.exists(CUSTOM_DIR):
    shutil.rmtree(CUSTOM_DIR)
print(f'[*] GitHub Custom 저장소 클론 중: {GITHUB_REPO_URL}')
!git clone {GITHUB_REPO_URL} {CUSTOM_DIR}

# 파이프라인 핵심 스크립트 복사
for item in ['architectural_cad_pipeline.py', 'glb_to_cad.py', 'multiview_utils']:
    src_path = os.path.join(CUSTOM_DIR, item)
    dest_path = os.path.join('/kaggle/working', item)
    if os.path.exists(dest_path):
        if os.path.isdir(dest_path):
            shutil.rmtree(dest_path)
        else:
            os.remove(dest_path)
    
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest_path)
    else:
        shutil.copy(src_path, dest_path)
print('[+] Custom 건축 CAD 파이프라인 핵심 연동 설치 성공!')

# C. 경량 의존성 패키지 설치 (Shape 전용으로 텍스처 의존성을 배제하여 엄청나게 빠른 환경 구성!)
%cd {REPO_DIR}/hy3dshape
!pip install -r requirements.txt -q
!pip install fast_simplification opencv-python trimesh -q

%cd /kaggle/working
print('[SUCCESS] 모든 건축 전용 경량화 라이브러리가 성공적으로 빌드되었습니다!')

In [ ]:
# 4. 업로드된 입력 빌딩 이미지/스케치 탐색
import glob

image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.webp',
                    '*.JPG', '*.JPEG', '*.PNG', '*.WEBP']
image_files = []
for ext in image_extensions:
    image_files.extend(glob.glob(f'/kaggle/input/**/{ext}', recursive=True))

if image_files:
    input_sketch_path = image_files[0]
    print(f'[+] 발견된 입력 대상 이미지: {input_sketch_path}')
    if len(image_files) > 1:
        print(f'  (총 {len(image_files)}개 이미지 발견, 첫 번째 항목 자동 할당)')
        for f in image_files:
            print(f'    - {f}')
else:
    print('[Warning] Kaggle input 폴더에 이미지를 찾을 수 없습니다.')
    print('[안내] 오른쪽 패널 > Add Data에서 디자인 스케치 이미지를 추가해주세요.')
    # 테스트용 기본 검정 더미 파일 생성 우회
    from PIL import Image
    input_sketch_path = '/kaggle/working/dummy_sketch.png'
    Image.new('RGB', (512, 512), (255, 255, 255)).save(input_sketch_path)
    print(f'  테스트 실행을 위해 화이트 더미 스케치를 임시 생성했습니다: {input_sketch_path}')

In [ ]:
# 5. 건축 CAD 및 디지털 제작 파이프라인 원클릭 복원 실행
import os, subprocess, gc
import torch

gc.collect()
torch.cuda.empty_cache()

# ═══════════════════════════════════════════════════════════════
# 📐 건축 특화 기하학 복원 및 디지털 제작 설정 매개변수 (User Parameters)
# ═══════════════════════════════════════════════════════════════

# ── 입력 모드 (Input Mode) ────────────────────────────────────────
# - 'single': 단 한 장의 입면도, 조감도, 컨셉 스케치로부터 정밀 복원 (강력 추천)
# - 'turnaround': 여러 각도(정면, 우측, 후면, 좌측)가 결합된 가로로 긴 턴어라운드 시트 파일인 경우
INPUT_MODE = 'single'           # 'single' 또는 'turnaround'
NUM_VIEWS  = 4                  # 'turnaround' 모드 시 가로 슬라이싱할 뷰 갯수 (4 또는 6)

# ── 3D 복원 디테일 해상도 ──────────────────────────────────────────
OCTREE_RESOLUTION = 512         # 메쉬 정밀도. 기본 512. T4 가용 한계치: 640
STEPS             = 50          # 디퓨전 연산 스텝 수. 50=기본, 100=극도 디테일(2배 느림)
GUIDANCE_SCALE    = 5.0         # 입력 스케치에 대한 형태 충실도 배율. 기본 5.0

# ── 지면 고정 장치 (Ground Locking) ──────────────────────────────────
LOCK_GROUND  = True             # True: 불안정한 바닥면 부위를 칼로 자르듯 평평하게 다듬고 Z=0 원점으로 스냅
GROUND_RATIO = 0.05             # 메쉬 하부에서 평평하게 압착할 슬라이스 두께 비율 (기본: 메쉬 전체 높이의 5%)

# ── 곡면 스무딩 최적화 (Mesh Smoothing) ────────────────────────────────
# - 'laplacian': 빠른 연산 및 깔끔한 형상 스무딩 필터 (기본)
# - 'taubin': 메쉬의 부피(Volume)의 수축 현상을 방지하면서 고주파 노이즈만 효율적으로 다듬는 고급 스무딩 필터
# - 'none': 원본 AI 생성 다면체 상태 유지
SMOOTHING_METHOD     = 'laplacian'
SMOOTHING_ITERATIONS = 10       # 스무딩 반복 적용 패스 횟수. (10회 권장, 높을수록 둥글고 부드러워짐)
SMOOTHING_LAMB       = 0.5      # 스무딩 반응 속도 강도 (0.0 ~ 1.0)

# ── 등고선 슬라이싱 단면 추출 (Contour Slicing) ────────────────────────
# CAD 도면 연동 및 레이저 커팅 와플 가공용 수평 등고 단면 경로를 대량 생성합니다.
SLICING_INTERVAL = 0.2          # Z축 슬라이싱 간격 (실제 스케일 단위, 예: 0.2m 간격 등)
                                # 0 이하 입력 시 등고 단면 추출 비활성화

# ── CAD 포맷 정규화 및 크기 ──────────────────────────────────────────
SCALE_FACTOR = 'auto'           # 'auto'(자동 감지 스케일링) 또는 수동 스케일링 배율 float
TARGET_UNIT  = 'mm'             # 디지털 패브리케이션 장비 가공 실제 단위 ('mm', 'cm', 'm')
DECIMATE     = 0.0              # 다면체 경량화 배율 (0.0: 원본 유지, 0.5: 용량 50% 축소 경량화)

# ═══════════════════════════════════════════════════════════════

OUTPUT_DIR = '/kaggle/working/output_architectural'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 파이프라인 CLI 인자 동적 생성
cmd = [
    'python', 'architectural_cad_pipeline.py',
    '--device', 'cuda:0',
    '--output_dir', OUTPUT_DIR,
    '--octree_resolution', str(OCTREE_RESOLUTION),
    '--steps', str(STEPS),
    '--guidance_scale', str(GUIDANCE_SCALE),
    '--ground_ratio', str(GROUND_RATIO),
    '--smoothing_method', SMOOTHING_METHOD,
    '--smoothing_iterations', str(SMOOTHING_ITERATIONS),
    '--smoothing_lamb', str(SMOOTHING_LAMB),
    '--slicing_interval', str(SLICING_INTERVAL),
    '--scale', str(SCALE_FACTOR),
    '--unit', TARGET_UNIT,
    '--decimate', str(DECIMATE),
]

if INPUT_MODE == 'single':
    cmd += ['--image', input_sketch_path]
else:
    cmd += ['--sheet', input_sketch_path, '--num_views', str(NUM_VIEWS), '--use_rembg']

if LOCK_GROUND:
    cmd.append('--lock_ground')

print("[*] 건축 전용 CAD/제작 기하학 복원 파이프라인 가동을 개시합니다...")
print(f"[Run CLI]: {' '.join(cmd)}\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8')
allowed_prefixes = ('[*]', '[+]', '[SUCCESS]', '[Error]', '[Warning]', '---', '===', 'Hunyuan3D:', '[shape]', '[multiview]', '[Ground Locking]', '[Smoothing]', '[Slicing]', '[CAD]')
for line in proc.stdout:
    clean_line = line.strip()
    if clean_line.startswith(allowed_prefixes) or any(kw in clean_line for kw in ['SUCCESS', 'Error', 'Warning']):
        print(clean_line, flush=True)
proc.wait()

if proc.returncode != 0:
    print(f"\n[Error] 파이프라인 비정상 중단됨 (code {proc.returncode}). 로그를 확인하십시오.")
else:
    print("\n🎉 [SUCCESS] 건축용 3D CAD/제작 파이프라인의 모든 복원 프로세스가 정상 완료되었습니다!")

In [ ]:
# 6. 전체 결과물 다운로드용 원클릭 ZIP 아카이브 빌드
import os, glob, shutil

zip_name = '/kaggle/working/architectural_cad_fabrication_results'
if os.path.exists(OUTPUT_DIR):
    print('[*] 등고선 파일(SVG/DXF) 및 CAD 메쉬 파일(STL/PLY/OBJ)들을 압축 패키징 중...')
    shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)
    print(f'[+] ZIP 아카이브 생성 성공: {zip_name}.zip ({os.path.getsize(zip_name + ".zip") / 1024 / 1024:.2f} MB)')

print('\n📂 복원 완료된 정밀 에셋 리스트:')
output_files = glob.glob(f'{OUTPUT_DIR}/**/*.*', recursive=True)
for f in sorted(output_files):
    if os.path.isfile(f):
        size_mb = os.path.getsize(f) / 1024 / 1024
        rel_path = os.path.relpath(f, '/kaggle/working')
        print(f'  - 📦 {rel_path}  ({size_mb:.2f} MB)')

print('\n' + '='*60)
print('📢 [다운로드 권장 방법]')
print('1. 대량 단면 파일이 들어있으므로, 우측 패널의 Output 섹션에서 [architectural_cad_fabrication_results.zip] 아카이브를 하나만 즉시 다운로드하여 사용하시면 매우 편합니다!')
print('2. 개별 파일들은 Rhino 3D, AutoCAD, 3D 프린터 슬라이서(Cura, PrusaSlicer)에 바로 드래그앤드롭하여 사용 가능합니다.')
print('='*60)